# SafeRide Vision — Google Colab setup

Fetch code from GitHub, mount Drive, install repository requirements, stage required weights from Drive or upload them, verify files, then start the backend.

Set `REPO_URL` below to the GitHub URL for this project. Keep missing checkpoints in the configured Drive paths, or upload them when prompted.

In [ ]:
# Set your repository URL and branch.
REPO_URL = "https://github.com/YOUR_ACCOUNT/YOUR_REPOSITORY.git"  # replace this
BRANCH = "main"
REPO_DIR = "/content/deepSort"

import subprocess
from pathlib import Path
if "YOUR_ACCOUNT" in REPO_URL or "YOUR_REPOSITORY" in REPO_URL:
    raise ValueError("Replace REPO_URL with the GitHub URL for this project.")
repo = Path(REPO_DIR)
if (repo / ".git").exists():
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    if repo.exists() and any(repo.iterdir()):
        raise RuntimeError(f"{REPO_DIR} exists and is not a Git checkout")
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
print("Code ready:", REPO_DIR)

In [ ]:
# Mount Drive and configure the external checkpoint locations.
from google.colab import drive
drive.mount("/content/drive")
DRIVE_WEIGHT_PATHS = {
    "reid": "/content/drive/MyDrive/Motor-Bikes-Data/Replace-files/model.pth.tar-50",
    "orientation": "/content/drive/MyDrive/Motor-Bikes-Data/oreintation detector.pt",
    "blink": "/content/drive/MyDrive/Motor-Bikes-Data/blink model weights/best_model.pth",
    "bike_plate": "/content/drive/MyDrive/Motor-Bikes-Data/no plates/best (4).pt",
}

In [ ]:
# Install the requirements from the GitHub checkout.
%cd /content/deepSort
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt

## Stage and verify weights

Four external weights are required by current code: ReID, orientation, blink, and the bike/plate detector (currently configured as one shared checkpoint). If Drive paths are missing, Colab asks you to upload the exact filenames below. The YOLO bike, mirror, and indicator weights must exist in the GitHub checkout.

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

specs = [
    ("reid", "model.pth.tar-50", Path("/content/drive/MyDrive/Motor-Bikes-Data/Replace-files/model.pth.tar-50")),
    ("orientation", "oreintation detector.pt", Path("/content/drive/MyDrive/Motor-Bikes-Data/oreintation detector.pt")),
    ("blink", "best_model.pth", Path("/content/drive/MyDrive/Motor-Bikes-Data/blink model weights/best_model.pth")),
    ("bike_plate", "best (4).pt", Path("/content/drive/MyDrive/Motor-Bikes-Data/no plates/best (4).pt")),
]
missing = []
for key, filename, destination in specs:
    source = Path(DRIVE_WEIGHT_PATHS[key]).expanduser()
    if source.is_file():
        destination.parent.mkdir(parents=True, exist_ok=True)
        if source.resolve() != destination.resolve():
            shutil.copy2(source, destination)
        print(f"Found {key}: {source}")
    elif destination.is_file():
        print(f"Already staged {key}: {destination}")
    else:
        missing.append((key, filename, destination))

if missing:
    print("Upload all missing files using these exact filenames:")
    for key, filename, _ in missing:
        print(f"  {key}: {filename}")
    uploaded = files.upload()
    for key, filename, destination in missing:
        if filename not in uploaded:
            raise FileNotFoundError(f"Upload missing file with exact name: {filename}")
        destination.parent.mkdir(parents=True, exist_ok=True)
        destination.write_bytes(uploaded[filename])
        print(f"Staged {key}: {destination}")

local_weights = [
    Path(REPO_DIR) / "yolov10m.pt",
    Path(REPO_DIR) / "mirror-indicator-yolov10m/weights/bestM.pt",
    Path(REPO_DIR) / "indicator-yolov12m/weights/best.pt",
]
for path in local_weights:
    if not path.is_file():
        raise FileNotFoundError(f"Missing repository weight: {path}")
    print(f"Found local weight: {path} ({path.stat().st_size:,} bytes)")

for key, filename, path in specs:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f"Missing or empty {key} checkpoint: {path}")
    print(f"Verified {key}: {path} ({path.stat().st_size:,} bytes)")

## Start the backend

If using ngrok, enter your own token privately when prompted. The token is not saved in this notebook.

In [ ]:
# Optional ngrok auth. Leave blank to skip.
from getpass import getpass
from pyngrok import conf

token = getpass("Ngrok auth token (leave blank to skip): ")
if token.strip():
    conf.get_default().auth_token = token.strip()

%cd /content/deepSort
!python backend_server.py